In [18]:
%env PGPASSWORD=5J8DhII0RRsPW1

env: PGPASSWORD=5J8DhII0RRsPW1


In [94]:
import pandas as pd
from constants.db_connections import ENGINE_READ_ONLY
from constants.db_names.names import data 
input = [
    'LV7001885838',
    'LV7008887632',
    'LV7009026245',
    'LV7008886642',
    'LV7001884279',
    'LV7001884009',
    'LV7001884028',
    'test'
]

query = lambda library_ids: f'''
WITH filtered_table AS (
    select distinct library_id, robot_sample_id from edna_wetlab_report 
    where library_id in {library_ids})
select filtered_table.library_id, ers.robot_sample_id, eas.archive_sample_id, fs.field_sample_id
FROM filtered_table
left JOIN edna_robot_sample ers on filtered_table.robot_sample_id = ers.robot_sample_id
left join edna_archive_sample eas on ers.archive_sample_id = eas.archive_sample_id
left join field_sample fs on eas.field_sample_id = fs.field_sample_id
order by library_id;
'''

df = pd.read_sql(query(tuple(input)), ENGINE_READ_ONLY)
missing_lib_ids = set(input) - set(df[data.edna_wetlab_report.library_id()])
if len(missing_lib_ids) > 0:
    # Send mail to Xihan
    print(missing_lib_ids)
if df.isna().any().any():
    pass
    # Send mail to jesper with the dataframe

{'test'}


In [95]:
df

,library_id,robot_sample_id,archive_sample_id,field_sample_id
0,LV7001884009,None,None,None
1,LV7001884028,None,None,None
2,LV7001884279,lv7001308345,None,None
3,LV7001885838,None,None,None
4,LV7008886642,LV6000199939,LV3006006216,None
5,LV7008887632,None,None,None
6,LV7009026245,None,None,None


In [19]:

    
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

# Email details
sender_email = "you@yourdomain.com"
receiver_email = "recipient@example.com"
subject = "Email with Attachment"
body = "This email contains an attachment sent via Postfix."

# File to attach
file_path = "/path/to/your/file.txt"
file_name = "file.txt"  # Name to appear in the email

# Create the email
message = MIMEMultipart()
message["From"] = sender_email
message["To"] = receiver_email
message["Subject"] = subject

# Attach the email body
message.attach(MIMEText(body, "plain"))

# Attach the file
with open(file_path, "rb") as file:
    part = MIMEBase("application", "octet-stream")
    part.set_payload(file.read())

# Encode the file for email
encoders.encode_base64(part)
part.add_header(
    "Content-Disposition",
    f"attachment; filename={file_name}",
)

message.attach(part)

# Send the email via Postfix
try:
    with smtplib.SMTP("localhost", 25) as server:  # Postfix typically listens on port 25
        server.send_message(message)
        print("Email sent successfully with attachment!")
except Exception as e:
    print(f"Error: {e}")



In [83]:
for i, column in enumerate(id_columns):
    
    print(column)
    print(df.query(f'{column}.isna()').iloc[:, :df.columns.get_loc(column)])

robot_sample_id
     library_id
0  LV7001884009
1  LV7001884028
3  LV7001885838
5  LV7008887632
6  LV7009026245
archive_sample_id
     library_id robot_sample_id
0  LV7001884009            None
1  LV7001884028            None
2  LV7001884279    lv7001308345
3  LV7001885838            None
5  LV7008887632            None
6  LV7009026245            None
field_sample_id
     library_id robot_sample_id archive_sample_id
0  LV7001884009            None              None
1  LV7001884028            None              None
2  LV7001884279    lv7001308345              None
3  LV7001885838            None              None
4  LV7008886642    LV6000199939      LV3006006216
5  LV7008887632            None              None
6  LV7009026245            None              None


In [66]:
df.columns.get_loc('robot_sample_id')

1

In [63]:
for index, row in df.iterrows():
    print(row[1])

None
None
lv7001308345
None
LV6000199939
None
None


C:\Users\glj523\AppData\Local\Temp\ipykernel_19264\3319070448.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(row[1])


In [32]:
for column in df.columns:
    if df[column].isna().any():
        print(column)
        # Send mail to responsible uploader

robot_sample_id
archive_sample_id
field_sample_id


True

In [13]:
tables = {}
for table in list(data.__dict__.keys()):
    if not table.startswith('_'):
        q = f'select * from {data()}.{table}'
        df = pd.read_sql(q, con=ENGINE_READ_ONLY)
        tables[table] = df

In [15]:
data.__dict__

mappingproxy({'__module__': 'constants.db_names.names',
              '_db_id': 1,
              'edna_robot_sample': constants.db_names.names.data.edna_robot_sample,
              'age_depth_model': constants.db_names.names.data.age_depth_model,
              'master_depth': constants.db_names.names.data.master_depth,
              'field_sample_types': constants.db_names.names.data.field_sample_types,
              'cgg_animal_plant': constants.db_names.names.data.cgg_animal_plant,
              'country_ocean': constants.db_names.names.data.country_ocean,
              'field_sample_environment_types': constants.db_names.names.data.field_sample_environment_types,
              'edna_wetlab_report': constants.db_names.names.data.edna_wetlab_report,
              'flowcell': constants.db_names.names.data.flowcell,
              'cgg_sediment_water': constants.db_names.names.data.cgg_sediment_water,
              'adna_wetlab_report': constants.db_names.names.data.adna_wetlab_report,
 

In [ ]:
how = 'outer'

merge_keys = {'field_sample': 'field_sample_id',
              'edna_archive_sample': ['archive_sample_id', 'field_sample_id'],
              'edna_robot_sample': ''}

tables['field_sample']['field_sample_id'] = tables['field_sample_id'].str.lower()


merged = (
    tables['field_sample']
    .merge(tables['edna_archive_sample'], on='field_sample_id', how=how)
    .merge(tables['edna_archive_sample'], on='field_sample_id', how=how)
    .merge(tables['edna_archive_sample'], on='field_sample_id', how=how)
    .merge(tables['edna_archive_sample'], on='field_sample_id', how=how)
    .merge(tables['edna_archive_sample'], on='field_sample_id', how=how)
    .merge(tables['edna_archive_sample'], on='field_sample_id', how=how)
)